In [1]:
import numpy as np

In [4]:
a = np.arange(5)
print(a)

[0 1 2 3 4]


In [ ]:
import numpy as np
import glob
import os
import re
import sys

# This finds the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

# --- THIS IS THE LINE YOU ARE MISSING ---
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
# --- ADD THAT LINE! ---

print(f"Project root added to path: {PROJECT_ROOT}")


In [ ]:
np.sqrt(2*1500*2*1e-7)

In [ ]:
from simulation.solvers.spatial_process import simul_initialize, simul_run
from simulation.solvers.rate_conversions import calculate_kappas


from pathlib import Path
import os

In [ ]:
""" Main execution block containing all physics parameters. """
###### ================================== 1. parameter setting =====================================
L = 2. # cubic box length

diff_scale = 1500. 
DA = 1. 
DB = 1. 
DX = 1.  
DX2 = 1. 

##### There are 6 reactions but only 4 sigma values
##### because the reactions B <-> X involve no sigma value
sigmas = np.array((1., 1., 1., 1.)) * 0.1 # sigma_r1f, sigma_r1b, sigma_r2f, sigma_r2b

box_shape = np.array((L, L, L,))

##### the Part to change freely for the corresponding simulation
# Schloegl's model reaction rates
k = np.array((0.15, 0.025, 5.75, 25.))
print("Reaction rates for bistable schloegl's model: ",k)
# full model reaction rates
ls = np.array((1.5, 1500., 150., 25., 5.75, 25.))
print("Reaction rates for bistable full model: ",ls)
    

In [ ]:
range = [0.2, 0.5, 1.0, 2.0, 5.0, 10.0]

In [ ]:
for i in range:
    print(f"Current diffusion coeff is: {diff_scale/i}.")
    diffusions = np.array((DX, DX2, DA, DB)) * diff_scale / i
    print(f"Diffusions are : {diffusions}.")
    kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)

In [ ]:
diffusions = np.array((DX, DX2, DA, DB)) * 1500
kappas = calculate_kappas(ls, diffusions[2], diffusions[0], diffusions[1], sigmas)

In [ ]:
def l1_plus_formula(kappa_1_plus, D, sigma):
     # Calculate the term inside the tanh function
    sqrt_term = np.sqrt(kappa_1_plus / (2 * D))
    
    # Calculate the Left-Hand Side (LHS) of the equation
    # This is the expression for the effective rate l_1^+
    tanh_val = np.tanh(sigma * sqrt_term)
    lhs = 4 * np.pi * D * (sigma - (1 / sqrt_term) * tanh_val)
    return lhs

def calculate_l2_rates(kappa_2_plus, kappa_2_minus, DA, DX, DX2, sigma_3):
    if kappa_2_plus <= 0 or kappa_2_minus <= 0: 
        return np.inf, np.inf
    
    alpha_sq = kappa_2_plus / (DX2 + DA) + kappa_2_minus / (DX2 + DX)
    alpha = np.sqrt(alpha_sq)
    common_factor = 4 * np.pi * (1 / alpha_sq) * (sigma_3 - np.tanh(alpha * sigma_3) / alpha)
    l2_plus = kappa_2_plus * common_factor
    l2_minus = kappa_2_minus * common_factor

    return l2_plus, l2_minus

In [ ]:
D = [20, 500, 750, 1000, 1500, 1600]
print(f"Kappas are {kappas}")
for k, _ in enumerate(D):
    print(f" ----- Current D is {D[k]} ----- ")
    print("Calculated l is:")
    l1p = l1_plus_formula(kappas[0], D[k], sigma=sigmas[0])
    l2p, l2m = calculate_l2_rates(kappas[2], kappas[3], D[k], D[k], D[k], sigmas[0])
    print(f"l1p:{l1p:.2f}, l2p:{l2p:.2f}, l2m:{l2m:.2f}")